In [57]:
from __future__ import annotations

import random
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from collections.abc import Sequence

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

try:
    from transformers import AutoModel, AutoTokenizer
except ImportError as e:  # pragma: no cover
    raise ImportError(
        "This script needs `transformers` and `torch`. "
        "Install with: pip install torch transformers --break-system-packages"
    ) from e


MODEL_NAME = "jihuai/bert-ancient-chinese"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Pair = tuple[str, str]

# Data structures

In [81]:
@dataclass
class Example:
    """One occurrence of a character pair inside a sentence / context window."""

    sentence: str
    i: int  # character index of the first component within `sentence`
    j: int  # character index of the second component within `sentence`
    pair: Pair = field(default=None)  # type: ignore[assignment]
    label: int = None  # 1 = collocation, 0 = non-collocation, None = unknown

    def __post_init__(self):
        if self.pair is None:
            self.pair = (self.sentence[self.i], self.sentence[self.j])
        assert self.sentence[self.i] == self.pair[0], "index i does not match pair[0]"
        assert self.sentence[self.j] == self.pair[1], "index j does not match pair[1]"

RawExample = str | dict[str, str | int]

# A raw example is either:
#   - a plain sentence string (positions are auto-located), or
#   - a dict like {"sentence": "...", "i": 3, "j": 5}

def _find_closest_occurrence(sentence: str, c1: str, c2: str) -> tuple[int, int]:
    """Find the (i, j) occurrence of c1, c2 in `sentence` with the smallest
    character distance between them. Returns None if no occurrence exists."""
    idx1 = [k for k, ch in enumerate(sentence) if ch == c1]
    idx2 = [k for k, ch in enumerate(sentence) if ch == c2]
    best = None
    best_dist = None
    for i in idx1:
        for j in idx2:
            if i == j:
                continue
            dist = abs(i - j)
            if best_dist is None or dist < best_dist:
                best_dist = dist
                best = (i, j)
    return best

def _find_begin_end(sentence: str, c1: str, c2: str) -> tuple[int, int]:
    """Find the first c1 and last c2 in a sentence."""
    i = sentence.index(c1)
    j = next(k for k in reversed(range(len(sentence))) if sentence[k] == c2)
    return (i, j)


def build_examples(
    pair_examples: dict[Pair, Sequence[RawExample]],
    label: int,
) -> list[Example]:
    """Turn {pair: [sentence_or_dict, ...]} into a flat list of Example objects.

    If a raw example is a plain string, the character positions are located
    automatically. If you already know the
    exact positions, pass dicts with explicit 'i' / 'j' instead.
    """
    out: list[Example] = []
    skipped = 0
    
    for (c1, c2), raws in pair_examples.items():
        c1 = c1.replace("尙", "尚").replace("竝", "並").replace("眞", "真").replace("愼", "慎")
        for raw in raws:
            if isinstance(raw, dict):
                sentence = raw["sentence"]
                i, j = raw["i"], raw["j"]
            else:
                
                sentence = raw
                try:
                    found = _find_begin_end(sentence, c1, c2)
                except ValueError:
                    found = None
                    print(f"({c1}, {c2}): {sentence}")
                if found is None:
                    skipped += 1
                    continue
                i, j = found
            out.append(Example(sentence=sentence, i=i, j=j, pair=(c1, c2), label=label))
    if skipped:
        warnings.warn(f"Skipped {skipped} example(s) where the pair could not be located.")
    return out


而
了
而你去了而且寫了甚麼
0


[Example(sentence='而你去了而且寫了甚麼', i=0, j=7, pair=('而', '了'), label=0)]

In [60]:
import pandas as pd
confident_pairs_df = pd.read_csv("../output/03_measures/exp1_selected_df_2tail.csv")

confident_pairs_df_repel = confident_pairs_df.loc[confident_pairs_df["pmi"] < 0]
confident_non_collocations = [(dc, sfp) for (i, (dc, sfp, n, pmi)) in confident_pairs_df_repel.iterrows()]
confident_pairs_df_attract = confident_pairs_df.loc[confident_pairs_df["pmi"] > 0]
confident_collocations = [(dc, sfp) for (i, (dc, sfp, n, pmi)) in confident_pairs_df_attract.iterrows()]

In [61]:
for i in confident_pairs_df_repel:
    print(i)

dc
sfp
n
pmi


In [62]:
with open(r"..\output\01_extract_pairs\combosDF.csv", "r", encoding="utf-8") as in_file:
    combo_df = pd.read_csv(in_file)

In [63]:

confident_collocation_sentences = dict()
confident_non_collocation_sentences = dict()
non_confident_sentences = dict()

import tqdm
for (i, item) in tqdm.tqdm(combo_df.iterrows()):
    if item["dc"] != item["dc"]:
        continue
    if item["sfp"] != item["sfp"]:
        continue

    curr_pair = (item["dc"], item["sfp"])
    if curr_pair in confident_collocations:
        confident_collocation_sentences[curr_pair] =\
            confident_collocation_sentences.get(curr_pair, []) + [item["sentence"]]

    elif curr_pair in confident_non_collocations:
        confident_non_collocation_sentences[curr_pair] =\
            confident_non_collocation_sentences.get(curr_pair, []) + [item["sentence"]]

    else:
        non_confident_sentences[curr_pair] =\
            non_confident_sentences.get(curr_pair, []) + [item["sentence"]]

56643it [00:00, 78226.71it/s]


In [64]:
combo_df

,sfp,dc,docID,paragraphID,sentenceID,sentence
0,NaN,NaN,KR1d0052_001,0,KR1d0052_001_title,曲禮上
1,NaN,NaN,KR1d0052_001,1,KR1d0052_001_par1_1-3,曲禮曰
2,NaN,安,KR1d0052_001,1,KR1d0052_001_par1_10-12,安定辭
3,哉,NaN,KR1d0052_001,1,KR1d0052_001_par1_13-15,安民哉
4,NaN,毋,KR1d0052_001,1,KR1d0052_001_par1_4-6,毋不敬
...,...,...,...,...,...,...
56638,NaN,以,KR2e0003_501,1,KR2e0003_501_par1_988-991,以令諸侯
56639,NaN,厚,KR2e0003_501,1,KR2e0003_501_par1_99-102,傷者厚養
56640,NaN,NaN,KR2e0003_501,1,KR2e0003_501_par1_992-995,天下可定
56641,乎,何,KR2e0003_501,1,KR2e0003_501_par1_996-1002,何必以趙為先乎


# Train / validation split at the PAIR level (avoid leakage)

In [65]:
def split_pairs_train_val(
    pairs: Sequence[Pair], val_frac: float = 0.2, seed: int = 0
) -> tuple[list[Pair], list[Pair]]:
    pairs = list(pairs)
    rng = random.Random(seed)
    rng.shuffle(pairs)
    n_val = max(1, int(round(len(pairs) * val_frac))) if len(pairs) > 1 else 0
    val_pairs = pairs[:n_val]
    train_pairs = pairs[n_val:]
    return train_pairs, val_pairs


def filter_examples_by_pairs(examples: list[Example], pairs: Sequence[Pair]) -> list[Example]:
    pairset = set(pairs)
    return [ex for ex in examples if ex.pair in pairset]


In [66]:
def char_to_token_index(
    tokenizer, encoding, char_idx: int, sentence: str, is_fast: bool
) -> int:
    """Map a character index in `sentence` to the corresponding token index
    in `encoding['input_ids']` (which includes special tokens)."""
    if is_fast:
        offsets = encoding["offset_mapping"]
        for tok_idx, (start, end) in enumerate(offsets):
            if start == end:  # special tokens have (0, 0)
                continue
            if start <= char_idx < end:
                return tok_idx
        # fall through if not found (e.g. char_idx beyond truncation)
    # Slow-tokenizer / fallback path: count tokens in the prefix.
    # Ancient/classical Chinese BERT tokenizers are essentially char-level,
    # so this is a safe approximation even without offset mappings.
    prefix_ids = tokenizer(sentence[:char_idx], add_special_tokens=False)["input_ids"]
    return 1 + len(prefix_ids)  # +1 accounts for the leading [CLS]

In [67]:
class PairDataset(Dataset):
    def __init__(self, examples: list[Example], tokenizer, max_length: int = 128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_fast = getattr(tokenizer, "is_fast", False)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            ex.sentence,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=self.is_fast,
        )
        n_tokens = len(enc["input_ids"])
        pos1 = min(
            char_to_token_index(self.tokenizer, enc, ex.i, ex.sentence, self.is_fast),
            n_tokens - 1,
        )
        pos2 = min(
            char_to_token_index(self.tokenizer, enc, ex.j, ex.sentence, self.is_fast),
            n_tokens - 1,
        )
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "pos1": pos1,
            "pos2": pos2,
            "label": ex.label if ex.label is not None else -1,
        }



In [68]:
def make_collate_fn(pad_token_id: int):
    def collate_fn(batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        B = len(batch)
        input_ids = torch.full((B, max_len), pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((B, max_len), dtype=torch.long)
        pos1 = torch.zeros(B, dtype=torch.long)
        pos2 = torch.zeros(B, dtype=torch.long)
        labels = torch.zeros(B, dtype=torch.long)
        for k, x in enumerate(batch):
            L = len(x["input_ids"])
            input_ids[k, :L] = torch.tensor(x["input_ids"], dtype=torch.long)
            attention_mask[k, :L] = torch.tensor(x["attention_mask"], dtype=torch.long)
            pos1[k] = x["pos1"]
            pos2[k] = x["pos2"]
            labels[k] = x["label"]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pos1": pos1,
            "pos2": pos2,
            "labels": labels,
        }

    return collate_fn

# Model

In [70]:
class CollocationClassifier(nn.Module):
    """BERT encoder + a small MLP head over the two target-character
    representations. Set `freeze_backbone=True` to use BERT purely as a
    frozen feature extractor (only the head is trained); set it to False to
    fine-tune the whole model end-to-end."""

    def __init__(
        self,
        model_name: str = MODEL_NAME,
        freeze_backbone: bool = True,
        hidden_dim: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        h = self.backbone.config.hidden_size
        # features: emb1, emb2, |emb1-emb2|, emb1*emb2
        self.classifier = nn.Sequential(
            nn.Linear(h * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, input_ids, attention_mask, pos1, pos2):
        ctx = torch.no_grad() if self.freeze_backbone else torch.enable_grad()
        with ctx:
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            hidden = out.last_hidden_state  # (B, L, H)
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        emb1 = hidden[batch_idx, pos1]
        emb2 = hidden[batch_idx, pos2]
        feat = torch.cat([emb1, emb2, torch.abs(emb1 - emb2), emb1 * emb2], dim=-1)
        logits = self.classifier(feat)
        return logits

# Config

In [71]:

@dataclass
class Config:
    model_name: str = MODEL_NAME
    freeze_backbone: bool = True          # hyperparameter: freeze vs fine-tune
    hidden_dim: int = 256
    dropout: float = 0.1
    max_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    lr_head: float = 1e-3
    lr_backbone: float = 2e-5             # only used if freeze_backbone=False
    weight_decay: float = 0.01
    val_frac: float = 0.2
    seed: int = 0
    min_fraction: float = 0.5             # % of examples needed to flag a new collocation
    reference_agg: str = "mean"           # "mean" or "median" per-pair aggregation
    early_stopping_patience: int = 3


In [72]:

def train_classifier(
    train_examples: list[Example],
    val_examples: list[Example],
    cfg: Config,
    tokenizer=None,
) -> tuple[CollocationClassifier, "AutoTokenizer"]:
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    model = CollocationClassifier(
        model_name=cfg.model_name,
        freeze_backbone=cfg.freeze_backbone,
        hidden_dim=cfg.hidden_dim,
        dropout=cfg.dropout,
    ).to(DEVICE)

    collate_fn = make_collate_fn(tokenizer.pad_token_id)
    train_ds = PairDataset(train_examples, tokenizer, cfg.max_length)
    val_ds = PairDataset(val_examples, tokenizer, cfg.max_length)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    # class weights to handle imbalance
    labels = np.array([ex.label for ex in train_examples])
    n_pos = max(1, (labels == 1).sum())
    n_neg = max(1, (labels == 0).sum())
    total = n_pos + n_neg
    class_weights = torch.tensor(
        [total / (2 * n_neg), total / (2 * n_pos)], dtype=torch.float, device=DEVICE
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    if cfg.freeze_backbone:
        optimizer = torch.optim.AdamW(
            model.classifier.parameters(), lr=cfg.lr_head, weight_decay=cfg.weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(
            [
                {"params": model.backbone.parameters(), "lr": cfg.lr_backbone},
                {"params": model.classifier.parameters(), "lr": cfg.lr_head},
            ],
            weight_decay=cfg.weight_decay,
        )

    best_val_acc = -1.0
    best_state = None
    patience = 0

    for epoch in range(cfg.epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            logits = model(batch["input_ids"], batch["attention_mask"], batch["pos1"], batch["pos2"])
            loss = criterion(logits, batch["labels"])
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch["labels"].size(0)
        train_loss = total_loss / max(1, len(train_ds))

        val_acc, val_loss = evaluate(model, val_loader, criterion)
        print(
            f"epoch {epoch+1}/{cfg.epochs}  train_loss={train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= cfg.early_stopping_patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, tokenizer


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch["pos1"], batch["pos2"])
        loss = criterion(logits, batch["labels"])
        loss_sum += loss.item() * batch["labels"].size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
    return correct / max(1, total), loss_sum / max(1, total)


# Inference on nonconfident pairs + aggregation

In [ ]:

@torch.no_grad()
def predict_proba(
    model: CollocationClassifier,
    examples: list[Example],
    tokenizer,
    batch_size: int = 32,
    max_length: int = 128,
) -> list[float]:
    """Returns P(collocation) for each example, in the same order as `examples`."""
    model.eval()
    ds = PairDataset(examples, tokenizer, max_length)
    collate_fn = make_collate_fn(tokenizer.pad_token_id)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    probs: list[float] = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch["pos1"], batch["pos2"])
        p = torch.softmax(logits, dim=-1)[:, 1]
        probs.extend(p.cpu().tolist())
    return probs


@dataclass
class PairStats:
    pair: Pair
    n: int
    mean_prob: float
    median_prob: float
    frac_classified_collocation: float  # fraction with p >= 0.5


def aggregate_by_pair(examples: list[Example], probs: list[float]) -> dict[Pair, PairStats]:
    by_pair: dict[Pair, list[float]] = defaultdict(list)
    for ex, p in zip(examples, probs):
        by_pair[ex.pair].append(p)
    stats: dict[Pair, PairStats] = {}
    for pair, plist in by_pair.items():
        arr = np.asarray(plist, dtype=float)
        stats[pair] = PairStats(
            pair=pair,
            n=len(arr),
            mean_prob=float(arr.mean()),
            median_prob=float(np.median(arr)),
            frac_classified_collocation=float((arr >= 0.5).mean()),
        )
    return stats


# New collocation detection

In [ ]:

def compute_reference_threshold(
    val_collocation_stats: dict[Pair, PairStats], agg: str = "mean"
) -> float:
    """The confidence of the LEAST confident held-out confident-collocation
    pair. This is the floor a "remaining" pair's examples must clear to be
    counted as confidently-classified-as-collocation."""
    key = "mean_prob" if agg == "mean" else "median_prob"
    values = [getattr(s, key) for s in val_collocation_stats.values()]
    if not values:
        raise ValueError("No held-out confident-collocation pairs to calibrate against.")
    return min(values)


def detect_new_collocations(
    remaining_examples: list[Example],
    remaining_probs: list[float],
    reference_threshold: float,
    min_fraction: float = 0.5,
) -> dict[Pair, dict]:
    """For each remaining pair, compute the fraction of its examples whose
    P(collocation) is at least as high as `reference_threshold`. Flag the
    pair as a newly detected collocation if that fraction >= min_fraction."""
    by_pair: dict[Pair, list[float]] = defaultdict(list)
    for ex, p in zip(remaining_examples, remaining_probs):
        by_pair[ex.pair].append(p)

    results: dict[Pair, dict] = {}
    for pair, plist in by_pair.items():
        arr = np.asarray(plist, dtype=float)
        frac_confident = float((arr >= reference_threshold).mean())
        results[pair] = {
            "n": len(arr),
            "mean_prob": float(arr.mean()),
            "median_prob": float(np.median(arr)),
            "frac_at_least_reference_confidence": frac_confident,
            "frac_classified_collocation": float((arr >= 0.5).mean()),
            "is_detected_collocation": frac_confident >= min_fraction,
        }
    return results


## Pipeline

In [73]:

def run_pipeline(
    confident_collocations: dict[Pair, Sequence[RawExample]],
    confident_non_collocations: dict[Pair, Sequence[RawExample]],
    remaining_pairs: dict[Pair, Sequence[RawExample]],
    cfg: Config = None,
):
    cfg = cfg or Config()

    colloc_examples = build_examples(confident_collocations, label=1)
    non_colloc_examples = build_examples(confident_non_collocations, label=0)
    remaining_examples = build_examples(remaining_pairs, label=None)

    colloc_pairs = list(confident_collocations.keys())
    non_colloc_pairs = list(confident_non_collocations.keys())

    train_colloc_pairs, val_colloc_pairs = split_pairs_train_val(
        colloc_pairs, cfg.val_frac, cfg.seed
    )
    train_non_pairs, val_non_pairs = split_pairs_train_val(
        non_colloc_pairs, cfg.val_frac, cfg.seed + 1
    )

    train_examples = filter_examples_by_pairs(colloc_examples, train_colloc_pairs) + \
        filter_examples_by_pairs(non_colloc_examples, train_non_pairs)
    val_examples = filter_examples_by_pairs(colloc_examples, val_colloc_pairs) + \
        filter_examples_by_pairs(non_colloc_examples, val_non_pairs)

    print(f"Train: {len(train_examples)} examples over "
          f"{len(train_colloc_pairs)+len(train_non_pairs)} pairs")
    print(f"Val:   {len(val_examples)} examples over "
          f"{len(val_colloc_pairs)+len(val_non_pairs)} pairs")

    model, tokenizer = train_classifier(train_examples, val_examples, cfg)

    # Calibrate the reference threshold using held-out (validation) confident
    # collocations only -- these were NOT trained on, so their predicted
    # confidence is a fair estimate of generalization confidence.
    val_colloc_examples = filter_examples_by_pairs(colloc_examples, val_colloc_pairs)
    val_colloc_probs = predict_proba(model, val_colloc_examples, tokenizer, cfg.batch_size, cfg.max_length)
    val_colloc_stats = aggregate_by_pair(val_colloc_examples, val_colloc_probs)
    reference_threshold = compute_reference_threshold(val_colloc_stats, agg=cfg.reference_agg)
    print(f"Reference confidence threshold (least-confident held-out "
          f"confident collocation): {reference_threshold:.4f}")

    # Score the remaining (unlabeled) pairs.
    remaining_probs = predict_proba(model, remaining_examples, tokenizer, cfg.batch_size, cfg.max_length)
    detection_results = detect_new_collocations(
        remaining_examples, remaining_probs, reference_threshold, cfg.min_fraction
    )

    detected = {p: r for p, r in detection_results.items() if r["is_detected_collocation"]}
    print(f"Detected {len(detected)} new collocation(s) out of {len(detection_results)} candidate pairs.")

    return {
        "model": model,
        "tokenizer": tokenizer,
        "reference_threshold": reference_threshold,
        "val_confident_collocation_stats": val_colloc_stats,
        "remaining_pair_results": detection_results,
        "detected_collocations": detected,
    }



In [84]:
cfg = Config(
    freeze_backbone=True,   # flip to False to fine-tune the whole BERT
    epochs=5,
    batch_size=8,
    min_fraction=0.9,
)

results = run_pipeline(
    confident_collocation_sentences, confident_non_collocation_sentences, non_confident_sentences, cfg
)

for pair, info in results["detected_collocations"].items():
    print(pair, info)

Train: 2146 examples over 43 pairs
Val:   997 examples over 10 pairs


Loading weights: 100%|██████████| 197/197 [00:00<?, ?it/s]
BertModel LOAD REPORT from: jihuai/bert-ancient-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because m

epoch 1/5  train_loss=0.1340  val_loss=3.7144  val_acc=0.3992
epoch 2/5  train_loss=0.0405  val_loss=2.4567  val_acc=0.4493
epoch 3/5  train_loss=0.0348  val_loss=4.2328  val_acc=0.4644
epoch 4/5  train_loss=0.0396  val_loss=3.4556  val_acc=0.4223
epoch 5/5  train_loss=0.0339  val_loss=3.0379  val_acc=0.4453
Reference confidence threshold (least-confident held-out confident collocation): 0.0167
Detected 464 new collocation(s) out of 525 candidate pairs.
('毋', '焉') {'n': 3, 'mean_prob': 0.9976110259691874, 'median_prob': 0.9975590705871582, 'frac_at_least_reference_confidence': 1.0, 'frac_classified_collocation': 1.0, 'is_detected_collocation': True}
('非', '焉') {'n': 2, 'mean_prob': 0.9379976689815521, 'median_prob': 0.9379976689815521, 'frac_at_least_reference_confidence': 1.0, 'frac_classified_collocation': 1.0, 'is_detected_collocation': True}
('復', '矣') {'n': 6, 'mean_prob': 0.9989989598592123, 'median_prob': 0.9990468323230743, 'frac_at_least_reference_confidence': 1.0, 'frac_class